<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part B: Statistical Forecasting</h2>
<h2>Notebook B01: Exponential Smoothing Models</h2>
</div>

Part A ended with a target: on the temperature series, repeating last year's value gives a mean absolute
error of 1.74 °C, and any model worth its complexity has to beat that.

Exponential smoothing is the oldest family of methods that does. The idea is a single sentence — recent
observations matter more than old ones, with the weight decaying smoothly into the past — and the family
grows from there by adding one component at a time: a level, then a trend, then a season.

We build all three in turn, and finish by picking between them the way Notebook A06 says you should.

---

**Contents**

1. [Imports and Data Loading](#1.-Imports-and-Data-Loading)
2. [The Idea: Weights That Decay](#2.-The-Idea:-Weights-That-Decay)
3. [Simple Exponential Smoothing](#3.-Simple-Exponential-Smoothing)
4. [Holt's Linear Trend Method](#4.-Holt's-Linear-Trend-Method)
5. [Holt-Winters: Adding Seasonality](#5.-Holt-Winters:-Adding-Seasonality)
6. [Additive or Multiplicative](#6.-Additive-or-Multiplicative)
7. [Choosing a Model](#7.-Choosing-a-Model)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports-and-Data-Loading">1. Imports and Data Loading</h3>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing

import nb_config

sns.set_theme(style="whitegrid")

The main series is the CDC regional temperature data from Part A, split the same way, so the numbers here
can be compared directly with the baselines in Notebooks
[A05](./A05_Forecasting_baselines.ipynb) and [A06](./A06_Evaluating_models.ipynb).

Section 6 also needs the OPS electricity data. If you have not prepared either, run
[F01b](./F01b_Preparing_CDC_dataset.ipynb) and [F01a](./F01a_Preparing_OPS_datasets.ipynb) first.

In [ ]:
series = pd.read_parquet(nb_config.CDC_TEMP_PATH)["Brandenburg/Berlin"].asfreq("MS")

TEST_MONTHS = 24
SEASON_LENGTH = 12

train = series.iloc[:-TEST_MONTHS]
test = series.iloc[-TEST_MONTHS:]

print(f"Train: {train.index.min().date()} to {train.index.max().date()}  ({len(train)} months)")
print(f"Test:  {test.index.min().date()} to {test.index.max().date()}  ({len(test)} months)")

In [ ]:
def mean_absolute_error(actual, forecast):
    """The metric from Notebook A06, repeated here so this notebook stands alone."""
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))))


# The target to beat, from Notebook A05
last_cycle = train.iloc[-SEASON_LENGTH:].to_numpy()
seasonal_naive = pd.Series(
    [last_cycle[i % SEASON_LENGTH] for i in range(TEST_MONTHS)],
    index=test.index,
    name="Seasonal naive",
)

SEASONAL_NAIVE_MAE = mean_absolute_error(test, seasonal_naive)
print(f"Seasonal naive baseline: MAE = {SEASONAL_NAIVE_MAE:.2f} °C")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-The-Idea:-Weights-That-Decay">2. The Idea: Weights That Decay</h3>
</div>

A moving average of the last 12 months treats the observation from 12 months ago as just as important as
last month's, and the one from 13 months ago as worthless. That is a strange thing to believe.

Exponential smoothing replaces the cut-off with a decay. The forecast is a weighted average of every
observation ever seen, where the weight on an observation $k$ steps back is

$$\alpha(1-\alpha)^k$$

The single parameter $\alpha$, between 0 and 1, controls how fast the weights fall away. Since the
weights sum to 1, this really is an average: a **smoothing parameter**, not a scaling factor.

In [ ]:
lags = np.arange(0, 25)

fig, ax = plt.subplots(figsize=(12, 4.5))

for alpha, colour in [(0.1, "steelblue"), (0.3, "seagreen"), (0.6, "darkorange"), (0.9, "crimson")]:
    ax.plot(lags, alpha * (1 - alpha) ** lags, marker="o", markersize=4,
            color=colour, label=f"alpha = {alpha}")

ax.set_title("Weight given to an observation k steps in the past", fontsize=14, fontweight="bold")
ax.set_xlabel("k (steps back from the most recent observation)")
ax.set_ylabel("Weight")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The two extremes are worth holding on to, because they explain most of what the models do later:

- **A small $\alpha$** spreads the weight over a long history. The forecast is stable and slow to react,
  which is what you want when the series is noisy but its level is genuinely steady.
- **A large $\alpha$** concentrates almost everything on the last few observations. The forecast follows
  the data closely and is easily thrown by a single odd value.

At $\alpha = 1$ all the weight sits on the most recent observation, and the method collapses into the
naive forecast from Notebook A05. That is not a hypothetical, as the next section shows.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-Simple-Exponential-Smoothing">3. Simple Exponential Smoothing</h3>
</div>

**Simple exponential smoothing** (SES) applies that weighted average directly. It tracks one thing, the
**level** of the series, and its forecast for every future step is that level:

$$\ell_t = \alpha y_t + (1-\alpha)\ell_{t-1}, \qquad \hat{y}_{t+h} = \ell_t$$

Because the forecast is a single number repeated, SES suits series with no trend and no season. Our
temperature series has an emphatic season, so we already know how this ends. It is still worth running,
because *how* it fails is informative.

`statsmodels` chooses $\alpha$ for us by maximum likelihood when we call `.fit()`.

In [ ]:
ses = SimpleExpSmoothing(train).fit()
ses_forecast = ses.forecast(TEST_MONTHS)

print(f"Fitted alpha: {ses.params['smoothing_level']:.4f}")
print(f"Forecast:     {ses_forecast.iloc[0]:.2f} °C, repeated for {TEST_MONTHS} months")
print(f"Last observed value in the training data: {train.iloc[-1]:.2f} °C")
print()
print(f"MAE: {mean_absolute_error(test, ses_forecast):.2f} °C")

The optimiser pushed $\alpha$ all the way to **1**, and the forecast is exactly the last observed value.
SES has reproduced the naive forecast, right down to its MAE of 8.13 °C.

This is the model telling you it has nothing to offer. With no way to represent a season, the best it can
do to explain a strongly seasonal series is to track it as closely as possible, one step behind, which
means putting all the weight on the newest observation. A fitted $\alpha$ at the boundary is always worth
a second look: it usually means the model is missing a component the data needs.

Fixing $\alpha$ by hand shows what the parameter does to the smoothed path through the training data.

In [ ]:
recent = train["2015":]

fig, ax = plt.subplots(figsize=(14, 4.5))

ax.plot(recent, color="black", linewidth=1.2, alpha=0.7, label="Observed")

for alpha, colour in [(0.05, "steelblue"), (0.3, "seagreen"), (0.9, "crimson")]:
    smoothed = SimpleExpSmoothing(recent).fit(smoothing_level=alpha, optimized=False)
    ax.plot(recent.index, smoothed.fittedvalues, color=colour, linewidth=1.3,
            label=f"alpha = {alpha}")

ax.set_title("Simple exponential smoothing at different alphas", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

At $\alpha = 0.05$ the smoothed line is nearly flat: the level barely moves, so the seasonal swing is
treated as noise. At $\alpha = 0.9$ it tracks the series closely but lags a month behind, which is the
best a level-only model can manage. Neither is a forecast of the season; both are descriptions of it
after the fact.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-Holt's-Linear-Trend-Method">4. Holt's Linear Trend Method</h3>
</div>

**Holt's method** adds a second component. Alongside the level it tracks a **trend**, the amount the level
changes from one step to the next, with its own smoothing parameter $\beta$:

$$\ell_t = \alpha y_t + (1-\alpha)(\ell_{t-1} + b_{t-1})$$
$$b_t = \beta(\ell_t - \ell_{t-1}) + (1-\beta)b_{t-1}$$
$$\hat{y}_{t+h} = \ell_t + h\,b_t$$

The forecast is no longer flat: it is a straight line continuing at the current slope.

In [ ]:
holt = ExponentialSmoothing(train, trend="add").fit()
holt_forecast = holt.forecast(TEST_MONTHS)

print(f"alpha (level): {holt.params['smoothing_level']:.4f}")
print(f"beta  (trend): {holt.params['smoothing_trend']:.4f}")
print(f"MAE: {mean_absolute_error(test, holt_forecast):.2f} °C   (SES: "
      f"{mean_absolute_error(test, ses_forecast):.2f} °C)")

A modest improvement, and for the wrong reason. Holt's method has not discovered the warming trend of the
last century; with $\alpha$ and $\beta$ both near 0.88 it is reacting to the most recent few months, and
the "trend" it extrapolates is really the tail end of a seasonal swing.

That points at a well-known weakness. A linear trend continues forever, so over a long horizon the
forecast can walk off to values the series has never visited. The **damped** variant multiplies the trend
by a factor $\phi < 1$ at each step, so the extrapolated line flattens out.

In [ ]:
damped = ExponentialSmoothing(train, trend="add", damped_trend=True).fit()
damped_forecast = damped.forecast(TEST_MONTHS)

print(f"phi (damping): {damped.params['damping_trend']:.4f}")
print(f"MAE: {mean_absolute_error(test, damped_forecast):.2f} °C")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))

ax.plot(train["2021":], color="steelblue", linewidth=1.2, label="Train")
ax.plot(test, color="black", linewidth=1.8, label="Actual")
ax.plot(ses_forecast, color="darkorange", linewidth=1.4, linestyle="--", label="SES")
ax.plot(holt_forecast, color="crimson", linewidth=1.4, linestyle="--", label="Holt")
ax.plot(damped_forecast, color="purple", linewidth=1.4, linestyle="--", label="Holt (damped)")

ax.axvline(test.index[0], color="gray", linestyle="--", linewidth=1.0)
ax.set_title("Level-only and trend models on a seasonal series", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The picture makes the problem plain. SES is flat, Holt slopes away from the data, and the damped version
bends back towards flat. None of them has any notion that temperature comes back every year, so all three
are hopeless on this series regardless of how their parameters are tuned.

The missing component is the season.

**Exercise.** Fit Holt's method with `damped_trend=True` on the first 100 years of the series only (`series[:'1980']`) and forecast 240 months ahead. Plot it against the undamped version. How far apart are they after 20 years, and which looks more like a temperature series?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-Holt-Winters:-Adding-Seasonality">5. Holt-Winters: Adding Seasonality</h3>
</div>

**Holt-Winters** adds the third and final component: a **seasonal** term with its own smoothing parameter
$\gamma$, holding one seasonal factor per period of the cycle — twelve of them for monthly data.

The forecast becomes the level, plus the trend extrapolated forward, plus the seasonal factor belonging
to the month being forecast. This is the model the whole family has been building towards, and the first
one with any chance against the seasonal naive baseline.

In [ ]:
holt_winters = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=SEASON_LENGTH,
).fit()

hw_forecast = holt_winters.forecast(TEST_MONTHS)

print(f"alpha (level):    {holt_winters.params['smoothing_level']:.4f}")
print(f"beta  (trend):    {holt_winters.params['smoothing_trend']:.4f}")
print(f"gamma (seasonal): {holt_winters.params['smoothing_seasonal']:.4f}")
print()
print(f"MAE: {mean_absolute_error(test, hw_forecast):.2f} °C")
print(f"Seasonal naive baseline: {SEASONAL_NAIVE_MAE:.2f} °C")

1.22 °C against the baseline's 1.74 °C: a 30% reduction in average error, and the first model in this
notebook that has earned its keep.

The fitted parameters are as interesting as the score. All three are small, and $\beta$ and $\gamma$ are
essentially zero. Read them as statements about the data:

- $\gamma \approx 0$ means the seasonal pattern does not need updating. The shape of the German year has
  been the same for 140 years, so the model estimates it once and leaves it alone.
- $\beta \approx 0$ means the same for the slope.
- $\alpha \approx 0.05$ means the level moves slowly, with each new observation nudging it only slightly.

A model that has to keep revising its seasonal factors is telling you the season is unstable. This one is
telling you the opposite.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(train["2021":], color="steelblue", linewidth=1.2, label="Train")
axes[0].plot(test, color="black", linewidth=1.8, label="Actual")
axes[0].plot(hw_forecast, color="seagreen", linewidth=1.6, linestyle="--", label="Holt-Winters")
axes[0].plot(seasonal_naive, color="darkorange", linewidth=1.2, linestyle=":", label="Seasonal naive")
axes[0].axvline(test.index[0], color="gray", linestyle="--", linewidth=1.0)
axes[0].set_title("Holt-Winters against the baseline it has to beat", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Temperature (°C)")
axes[0].legend(loc="upper left")

axes[1].plot(test.index, hw_forecast.values - test.values, marker="o", markersize=4,
             color="seagreen", label="Holt-Winters")
axes[1].plot(test.index, seasonal_naive.values - test.values, marker="s", markersize=4,
             color="darkorange", label="Seasonal naive")
axes[1].axhline(0, color="black", linewidth=1.0)
axes[1].set_title("Forecast errors", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Forecast - actual (°C)")
axes[1].legend(loc="upper left")

for ax in axes:
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The error panel shows where the gain comes from. The seasonal naive forecast inherits whatever was unusual
about last year, so a single warm month twelve months ago becomes an error today. Holt-Winters averages
the season over the whole record, which makes it far less hostage to one unrepresentative year.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-Additive-or-Multiplicative">6. Additive or Multiplicative</h3>
</div>

The seasonal component comes in two forms, and choosing between them is the one modelling decision
Holt-Winters really asks of you.

- **Additive**: the season is a fixed number of units, added on. July is 18 °C above the level, whatever
  the level happens to be.
- **Multiplicative**: the season is a percentage, multiplied in. July is 40% above the level, so the
  seasonal swing grows as the level grows.

The rule of thumb is to look at the series: if the seasonal swings get visibly wider as the series rises,
use multiplicative. A series built to behave that way makes the difference obvious.

In [ ]:
# A synthetic series whose seasonal amplitude grows with its level
months = np.arange(96)
synthetic = pd.Series(
    (100 + 3 * months) * (1 + 0.25 * np.sin(2 * np.pi * months / 12)),
    index=pd.date_range("2015-01-01", periods=96, freq="MS"),
    name="Synthetic",
)

synthetic_train = synthetic.iloc[:-12]
synthetic_test = synthetic.iloc[-12:]

results = {}
for kind in ["add", "mul"]:
    model = ExponentialSmoothing(
        synthetic_train, trend="add", seasonal=kind, seasonal_periods=12
    ).fit()
    results[kind] = model.forecast(12)
    print(f"seasonal={kind!r:>6}  AIC={model.aic:9.1f}  "
          f"MAE={mean_absolute_error(synthetic_test, results[kind]):.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))

ax.plot(synthetic_train, color="steelblue", linewidth=1.2, label="Train")
ax.plot(synthetic_test, color="black", linewidth=1.8, label="Actual")
ax.plot(results["add"], color="crimson", linewidth=1.4, linestyle="--", label="Additive season")
ax.plot(results["mul"], color="seagreen", linewidth=1.4, linestyle="--", label="Multiplicative season")

ax.axvline(synthetic_test.index[0], color="gray", linestyle="--", linewidth=1.0)
ax.set_title("A series whose seasonal swing grows with its level", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Value")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The multiplicative model reproduces the series **exactly**, because that is how the series was built. The
additive model has to settle for one fixed seasonal amplitude and is wrong at both ends: too wide early
on, too narrow later.

Real data is rarely this obliging. The Austrian electricity load below is strictly positive and clearly
seasonal, which is the textbook setting for multiplicative seasonality.

In [ ]:
ops = pd.read_parquet(nb_config.OPS_15M_PATH)

austrian_load = (
    ops[(ops["country"] == "AT") & (ops["measure"] == "actual_entsoe_transparency")]["value"]
    .tz_convert(None)          # work in naive local time
    .resample("MS").mean()     # monthly average load, in MW
    .dropna()
    .asfreq("MS")
)

load_train = austrian_load.iloc[:-12]
load_test = austrian_load.iloc[-12:]

print(f"{austrian_load.index.min().date()} to {austrian_load.index.max().date()}  "
      f"({len(austrian_load)} months)")
print(f"Range: {austrian_load.min():.0f} to {austrian_load.max():.0f} MW")

In [ ]:
for kind in ["add", "mul"]:
    model = ExponentialSmoothing(
        load_train, trend="add", seasonal=kind, seasonal_periods=12
    ).fit()
    forecast = model.forecast(12)
    error = mean_absolute_error(load_test, forecast)
    print(f"seasonal={kind!r:>6}  AIC={model.aic:7.1f}  MAE={error:6.1f} MW  "
          f"({error / load_test.mean():.1%} of the mean)")

On the real series the two are almost indistinguishable: a few MW apart on an average load of nearly
7,000, with AIC marginally preferring the multiplicative version and held-out MAE marginally preferring
the additive one. Disagreement that small is noise.

The reason is that Austrian consumption has no strong upward trend over these six years. Multiplicative
seasonality only differs from additive when the *level changes substantially*, because that is the only
time "20% of the level" and "a fixed 1,400 MW" part company. On a flat series they describe the same
thing.

There is also a hard constraint, not just a preference.

In [ ]:
try:
    ExponentialSmoothing(
        train, trend="add", seasonal="mul", seasonal_periods=SEASON_LENGTH
    ).fit()
except ValueError as error:
    print(f"ValueError: {error}")

print(f"\nColdest month in the training data: {train.min():.1f} °C")

Multiplicative seasonality needs strictly positive data, and for a good reason: multiplying a seasonal
factor by a level that passes through zero, or is negative, is meaningless. Temperature in Celsius is
exactly the kind of series this rules out, the same property that made the coefficient of variation
useless in Notebook A06.

So the decision procedure is short:

1. Any value at or below zero? **Additive** is your only option.
2. Seasonal swings visibly widening as the level rises? **Multiplicative**.
3. Otherwise? Fit both and compare. The difference is usually small, and the data will tell you.

**Exercise.** Take the Rossmann store sales from Notebook A04, aggregate to weekly totals for a single store, and fit both variants with `seasonal_periods=52`. Which does AIC prefer? Does the choice matter as much as picking the right seasonal period?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-Choosing-a-Model">7. Choosing a Model</h3>
</div>

We now have four models on the temperature series. Notebook A06 laid out how to choose between them:
information criteria to narrow the field, held-out error to confirm, and a baseline to make the numbers
mean something.

In [ ]:
candidates = {
    "SES (level only)": SimpleExpSmoothing(train),
    "Holt (+ trend)": ExponentialSmoothing(train, trend="add"),
    "Holt damped": ExponentialSmoothing(train, trend="add", damped_trend=True),
    "Holt-Winters (+ season)": ExponentialSmoothing(
        train, trend="add", seasonal="add", seasonal_periods=SEASON_LENGTH
    ),
    "Season, no trend": ExponentialSmoothing(
        train, seasonal="add", seasonal_periods=SEASON_LENGTH
    ),
}

rows = []
for name, model in candidates.items():
    fitted = model.fit()
    rows.append({
        "Model": name,
        "Parameters": len(fitted.params_formatted),
        "AIC": fitted.aic,
        "BIC": fitted.bic,
        "Test MAE": mean_absolute_error(test, fitted.forecast(TEST_MONTHS)),
    })

comparison = (
    pd.DataFrame(rows)
    .sort_values("AIC")
    .reset_index(drop=True)
    .assign(**{"vs baseline": lambda d: d["Test MAE"] / SEASONAL_NAIVE_MAE})
)

comparison.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))

ordered = comparison.sort_values("Test MAE")
colours = ["seagreen" if value < SEASONAL_NAIVE_MAE else "steelblue"
           for value in ordered["Test MAE"]]

ax.barh(ordered["Model"], ordered["Test MAE"], color=colours)
ax.axvline(SEASONAL_NAIVE_MAE, color="crimson", linestyle="--", linewidth=1.4,
           label=f"Seasonal naive baseline ({SEASONAL_NAIVE_MAE:.2f})")

for y, value in enumerate(ordered["Test MAE"]):
    ax.text(value + 0.12, y, f"{value:.2f}", va="center", fontsize=10)

ax.invert_yaxis()
ax.set_title("Held-out error, against the baseline", fontsize=14, fontweight="bold")
ax.set_xlabel("MAE (°C)")
ax.set_xlim(0, ordered["Test MAE"].max() * 1.18)
ax.legend(loc="lower right")
ax.grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Two models beat the baseline and three do not, and the split is exactly the one the theory predicts: the
models with a seasonal component win, the models without it lose. Adding a trend on top of the season buys
nothing here — AIC and BIC both prefer the simpler seasonal-only model, and the held-out errors agree to
within 0.02 °C.

That is the useful lesson of this notebook, and it generalises well beyond exponential smoothing. The gain
came from adding the component the data actually has. Tuning the parameters of a model that is missing
that component, as we did at length in sections 3 and 4, could never have closed the gap.

> **A note on names.** You will meet this family called **ETS**, for Error, Trend, Seasonal, written as
> triples like ETS(A,A,A) for the additive Holt-Winters model used here. `statsmodels` also provides
> `ETSModel`, a newer implementation with a proper statistical basis that gives you likelihood-based
> prediction intervals. We return to those in Notebook
> [B04](./B04_Probabilistic_forecasting.ipynb).

**Exercise.** Run the rolling-origin evaluation from Notebook A06 on Holt-Winters and the seasonal naive forecast, with `horizon=12` over 10 origins. Does Holt-Winters beat the baseline at every origin, or only on average? Given the spread you found in A06, how confident are you in the 30% improvement reported above?

In [ ]:
# Your solution here


---

Exponential smoothing describes a series through components you choose in advance: a level, perhaps a
trend, perhaps a season. The next family takes a different route, modelling a series through its own past
values and past errors, and letting the correlation structure of the data decide the form:
[B02 - ARIMA Models](./B02_ARIMA_models.ipynb).

**Solutions.** Worked answers to the 3 exercises above, with the reasoning behind them, are in
[B01_Exponential_smoothing_models_solutions.ipynb](../solutions/B01_Exponential_smoothing_models_solutions.ipynb). Try each one yourself first.
